In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/antorio/FlashVSR_plus
%cd FlashVSR_plus

In [ ]:
# install torch dari index pytorch
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130

# install sisanya dari pypi
!pip install -r requirements.txt

In [ ]:
!rm -rf ./models/FlashVSR-v1.1
!git lfs clone https://huggingface.co/JunhaoZhuang/FlashVSR-v1.1 ./models/FlashVSR-v1.1

In [ ]:
BASE_DIR = "/content/drive/MyDrive/c"
INPUT = "SR08-29-2020_9.mp4" # @param {type:"string"}
MODE = "tiny" # @param ["full", "tiny", "tiny-long"]
SCALE = 2
OUTPUT_DIR = BASE_DIR + "/enh_output"

In [ ]:
import os
import subprocess

CHUNK_DIR = "/content/chunks"
CHUNK_SECONDS = 20
MAX_SIZE = 640

os.makedirs(CHUNK_DIR, exist_ok=True)

subprocess.run([
    "ffmpeg",
    "-i", f"{BASE_DIR}/{INPUT}",

    # resize (berdasarkan sisi terpanjang)
    #"-vf", f"scale='if(gt(iw,ih),min({MAX_SIZE},iw),-2)':'if(gt(iw,ih),-2,min({MAX_SIZE},ih))'",

    # encode
    "-c:v", "libx264",
    "-preset", "fast",
    "-crf", "8",

    # keyframe biar split presisi
    "-force_key_frames", f"expr:gte(t,n_forced*{CHUNK_SECONDS})",

    # ❗ buang audio
    "-an",

    "-f", "segment",
    "-segment_time", str(CHUNK_SECONDS),
    "-reset_timestamps", "1",
    "-avoid_negative_ts", "make_zero",

    f"{CHUNK_DIR}/part_%04d.mp4"
], check=True)

print("Done resize + split (no audio)")

In [ ]:
import glob
import os

TEMP_DIR = "/content/results"
os.makedirs(TEMP_DIR, exist_ok=True)

chunks = sorted(glob.glob(f"{CHUNK_DIR}/part_*.mp4"))

for i, f in enumerate(chunks):
    print("=================================")
    print("Processing:", f)
    print("=================================")
    #!python run.py -i "{f}" -m {MODE} -s {SCALE} {TEMP_DIR}
    !python run.py -i "{f}" -m {MODE} -s {SCALE} --no-vae {TEMP_DIR}
    #!python run.py -i "{f}" -m {MODE} -s {SCALE} --tiled-dit --tiled-vae --tile-size 384 {TEMP_DIR}

In [ ]:
import subprocess
import glob
import os

files = sorted(glob.glob(f"{TEMP_DIR}/FlashVSR_*.mp4"))
list_path = "/content/list.txt"

with open(list_path, "w") as f:
  for v in files:
    f.write(f"file '{os.path.abspath(v)}'\n")

def get_encoder_params():
    test_cmd = [
        "ffmpeg",
        "-hide_banner",
        "-f", "lavfi",
        "-i", "nullsrc=s=16x16:d=1",
        "-c:v", "h264_nvenc",
        "-f", "null",
        "-"
    ]

    test = subprocess.run(test_cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

    if test.returncode == 0:
        print("Using GPU encoder: h264_nvenc")
        return [
            "-c:v", "h264_nvenc",
            "-preset", "p4",
            "-cq", "14"
        ]
    else:
        print("Fallback to CPU encoder: libx264")
        return [
            "-c:v", "libx264",
            "-preset", "medium",
            "-crf", "12"
        ]

encoder_params = get_encoder_params()

cmd = [
    "ffmpeg",
    "-y",
    "-f", "concat",
    "-safe", "0",
    "-i", list_path,
    *encoder_params,
    "-movflags", "+faststart",
    f"{OUTPUT_DIR}/FINAL_{INPUT}"
]

subprocess.run(cmd, capture_output=True, text=True)

print("Final video done")

In [ ]:
import shutil
import os

# hapus folder chunks
if os.path.exists(CHUNK_DIR):
    shutil.rmtree(CHUNK_DIR)
    print(f"Deleted: {CHUNK_DIR}")

# hapus folder results sementara
if os.path.exists(TEMP_DIR):
    shutil.rmtree(TEMP_DIR)
    print(f"Deleted: {TEMP_DIR}")

# hapus file list.txt
list_path = "/content/list.txt"
if os.path.exists(list_path):
    os.remove(list_path)
    print(f"Deleted: {list_path}")

print("Done cleanup")

In [ ]:
from google.colab import runtime
runtime.unassign()